In [0]:
# =========================
# Import libraries
# =========================
from pyspark.sql import functions as F
import re

# =========================
# Define paths
# =========================
source_path = "s3://db-ingestion-data/datasets"
target_schema = "01_prod_bronze.raw"

print("Starting ingestion process...")

# =========================
# Function to convert to snake_case
# (handles camelCase also)
# =========================
def to_snake_case(name):
    name = name.replace(".csv", "")
    name = re.sub(r'(.)([A-Z][a-z]+)', r'\1_\2', name)
    name = re.sub(r'([a-z0-9])([A-Z])', r'\1_\2', name)
    name = re.sub(r'[^a-zA-Z0-9]', '_', name)
    name = re.sub(r'_+', '_', name)
    return name.lower().strip('_')

# =========================
# Get list of files
# =========================
files = dbutils.fs.ls(source_path)
csv_files = [f.path for f in files if f.path.endswith(".csv")]

print("Number of CSV files found:", len(csv_files))

# =========================
# Loop through each file
# =========================
for file_path in csv_files:
    
    # Get file name
    file_name = file_path.split("/")[-1]
    
    # Convert file name to table name
    table_name = to_snake_case(file_name)
    full_table_name = f"{target_schema}.{table_name}"
    
    print("\nProcessing file:", file_name)
    print("Creating table:", full_table_name)
    
    # =========================
    # Read CSV
    # =========================
    df = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .load(file_path)
    
    print("Data loaded")
    
    # =========================
    # Convert column names to snake_case
    # =========================
    new_cols = [to_snake_case(col) for col in df.columns]
    df = df.toDF(*new_cols)
    
    print("Columns converted to snake_case")
    
    # =========================
    # Add metadata columns
    # =========================
    df = df.withColumn("ingestion_timestamp", F.current_timestamp()) \
           .withColumn("source_file", F.lit(file_name))
    
    print("Metadata columns added")
    
    # =========================
    # Write to Delta table
    # =========================
    df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .option("mergeSchema", "true") \
        .saveAsTable(full_table_name)
    
    print("Table saved:", full_table_name)

# =========================
# Done
# =========================
print("\nAll files ingested successfully")